# Embedding Models for Retrieval: watch a dense embedder learn to cluster paraphrases

Retrieval can only find what the **embedder** places near the query in vector space. Our chapter-1 lexical retriever scored only **recall@1 = 0.50** on paraphrased queries, because "liftoff date" shares no words with "was launched on." That's not a chunking bug — it's an **embedding** bug, and this notebook fixes it.

We build two embedders from scratch and measure the difference: a **sparse** bag-of-words embedder (matches only shared words → fails paraphrases) and a **dense** bi-encoder trained with the contrastive **InfoNCE** loss (learns to place paraphrases nearby even with no shared words). Then we confirm the lesson on a real pretrained model. We import the canonical functions from [`embedding_models.py`](embedding_models.py); the from-scratch demo is **seeded and deterministic**, so every number is identical on every run.

**By the end you'll have:** seen a sparse embedder fail paraphrases, trained a dense bi-encoder that separates them, watched the contrastive loss pull positives together, and confirmed it on all-MiniLM-L6-v2.

## Step 1 — set up: the toy paraphrase pairs

Everything runs on four `(query, paraphrased-passage)` pairs — same meaning, deliberately **few shared words**. Print them with their word overlap: three of the four pairs share *no* content words at all. That's the exact case a lexical embedder cannot handle and a dense one must.

In [1]:
import numpy as np

from embedding_models import (
    PARAPHRASE_PAIRS, DENSE_DIM, TEMPERATURE,
    tokenize, build_vocab, sparse_embed, bow_tensor,
    train_bi_encoder, similarity_matrix, diagonal_vs_offdiagonal, try_pretrained_demo,
)
import torch

print('numpy:', np.__version__, '| torch:', torch.__version__)
print(f'pairs: {len(PARAPHRASE_PAIRS)} | dense_dim: {DENSE_DIM} | temperature: {TEMPERATURE}\n')
for q, p in PARAPHRASE_PAIRS:
    shared = sorted(set(tokenize(q)) & set(tokenize(p)))
    print(f'  Q: {q!r}')
    print(f'  P: {p!r}')
    print(f'  shared content words: {shared or "NONE"}\n')

numpy: 2.4.6 | torch: 2.12.0
pairs: 4 | dense_dim: 16 | temperature: 0.07

  Q: 'how do I reset my password'
  P: 'I forgot my login credentials'
  shared content words: ['i', 'my']

  Q: 'the car will not start'
  P: "my automobile won't turn on"
  shared content words: NONE

  Q: 'what is the refund policy'
  P: 'how do I get my money back'
  shared content words: NONE

  Q: 'the flight was delayed'
  P: 'my plane departed late'
  shared content words: NONE



## Step 2 — the sparse embedder fails the paraphrases

A bag-of-words embedder has one slot per vocabulary word, nonzero only where that word appears, then L2-normalized. Build the cosine matrix of every query against every passage: the **diagonal** is paraphrase pairs (should be high), the **off-diagonal** is unrelated pairs (should be low). For the sparse model, both are near zero — it can't tell them apart.

In [2]:
queries = [q for q, _ in PARAPHRASE_PAIRS]
passages = [p for _, p in PARAPHRASE_PAIRS]
vocab = build_vocab(queries + passages)
print('vocabulary size:', len(vocab), 'tokens')

def sparse_fn(text):
    return sparse_embed(text, vocab)

sparse_sim = similarity_matrix(queries, passages, sparse_fn)
sparse_diag, sparse_off = diagonal_vs_offdiagonal(sparse_sim)
print(f'SPARSE — paraphrase (diagonal) sim: {sparse_diag:.3f}')
print(f'SPARSE — unrelated (off-diagonal) sim: {sparse_off:.3f}')
print(f'SPARSE — gap (want large): {sparse_diag - sparse_off:+.3f}')

# correctness BEFORE any claim: the sparse embedder must NOT separate paraphrases
assert sparse_diag - sparse_off < 0.2, 'sparse embedder cannot match paraphrases (few shared words)'

vocabulary size: 32 tokens
SPARSE — paraphrase (diagonal) sim: 0.091
SPARSE — unrelated (off-diagonal) sim: 0.082
SPARSE — gap (want large): +0.009


## Step 3 — train a dense bi-encoder with InfoNCE

Now the fix. The dense bi-encoder is a learned linear map (bag-of-words → 16-dim → L2-normalized), trained with the contrastive **InfoNCE** loss: within the batch, each query's positive is its paired passage and every *other* passage is a negative. Minimizing the loss pulls positives together and pushes negatives apart. Watch the loss fall toward 0.

In [3]:
model, losses = train_bi_encoder(PARAPHRASE_PAIRS, vocab)   # 800 seeded InfoNCE steps
print(f'InfoNCE loss: {losses[0]:.3f} (start) -> {losses[-1]:.4f} (end)')
print(f'trained {len(losses)} steps')

InfoNCE loss: 3.862 (start) -> 0.0000 (end)
trained 800 steps


## Step 4 — the dense embedder separates paraphrases

Re-build the cosine matrix with the *trained* dense encoder. Now the diagonal (paraphrases) is high and the off-diagonal (unrelated) is low — a wide gap the sparse model couldn't produce. The model learned, from the pairs alone, that the paraphrases belong together despite sharing no words.

In [4]:
def dense_fn(text):
    return model(bow_tensor(text, vocab).unsqueeze(0)).squeeze(0).detach().numpy()

dense_sim = similarity_matrix(queries, passages, dense_fn)
dense_diag, dense_off = diagonal_vs_offdiagonal(dense_sim)
print(f'DENSE — paraphrase (diagonal) sim: {dense_diag:.3f}')
print(f'DENSE — unrelated (off-diagonal) sim: {dense_off:.3f}')
print(f'DENSE — gap: {dense_diag - dense_off:+.3f}   (sparse gap was {sparse_diag - sparse_off:+.3f})')

# the headline assertions: dense separates paraphrases, and beats sparse on the gap
assert dense_diag > 0.4, 'dense must pull paraphrases together'
assert (dense_diag - dense_off) > (sparse_diag - sparse_off), 'dense must beat sparse on the gap'
print('\ndense separates paraphrases; sparse cannot:', bool((dense_diag - dense_off) > (sparse_diag - sparse_off)))

DENSE — paraphrase (diagonal) sim: 0.675
DENSE — unrelated (off-diagonal) sim: -0.220
DENSE — gap: +0.895   (sparse gap was +0.009)

dense separates paraphrases; sparse cannot: True


## Step 5 — confirm on a real pretrained model

The from-scratch demo proves the *mechanism*. A production embedder confirms it at scale: **all-MiniLM-L6-v2** is a 384-dim Sentence-BERT bi-encoder trained on hundreds of millions of pairs. We encode the same sentences and check the gap. (This step degrades gracefully — if the model isn't available it's skipped, so the notebook never hard-fails on a download.)

In [5]:
pre_sim = try_pretrained_demo(queries, passages)
if pre_sim is not None:
    pre_diag, pre_off = diagonal_vs_offdiagonal(pre_sim)
    print('PRETRAINED (all-MiniLM-L6-v2, 384-dim)')
    print(f'  paraphrase sim: {pre_diag:.3f} | unrelated sim: {pre_off:.3f} | gap: {pre_diag - pre_off:+.3f}')
    assert pre_diag - pre_off > 0.2, 'a real bi-encoder must separate paraphrases too'
    print('  real bi-encoder separates paraphrases:', bool(pre_diag - pre_off > 0.2))
else:
    print('pretrained model unavailable — the from-scratch demo above is the load-bearing lesson')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

PRETRAINED (all-MiniLM-L6-v2, 384-dim)
  paraphrase sim: 0.631 | unrelated sim: 0.160 | gap: +0.471
  real bi-encoder separates paraphrases: True


## Try it yourself

Before you change anything, **predict**: the dense encoder learned to separate the *four training pairs*. If you feed it a **brand-new** sentence built from words it saw during training (e.g. `'my vehicle will not turn on'`, mixing words from the car pair), will it land near the car cluster — or fail, because it only memorized the exact training sentences?

Then try it: encode a new sentence with `dense_fn(...)` and check its cosine to each training passage. *Hint:* because the encoder works on a **bag of words** and learned per-word directions, it generalizes to *recombinations* of known words — but it has **no signal** for words it never saw in training (out-of-vocabulary → zero contribution). That OOV blind spot is exactly why real embedders use **subword tokenization** and train on massive corpora, so almost no word is unseen.

> To go further, swap our toy for a real model: `SentenceTransformer('all-MiniLM-L6-v2').encode(...)` and compare *your own* query/passage pairs — the same bi-encoder + cosine, at production scale.

## What we saw

- **The sparse embedder couldn't see paraphrases** — paraphrase and unrelated similarities were both near zero (gap ≈ +0.01), because the pairs share almost no words.
- **Contrastive training fixed it** — the InfoNCE loss fell to ~0 while the dense encoder pulled paraphrases together and pushed unrelated text apart, opening a wide gap (≈ +0.90).
- **A real model confirmed the mechanism** — all-MiniLM-L6-v2 (384-dim) separated the same pairs (gap ≈ +0.47), exactly what the from-scratch demo taught, at production scale.
- **The embedder sets the geometry** — same sentences, same words; only the model changed, and that alone decided whether retrieval could find the match.

**What we skipped** (and where to read it): the **bi-encoder vs cross-encoder** tradeoff and pooling/normalization details; **asymmetric prefixes** (E5 `query:`/`passage:`), domain mismatch, and max-length truncation; **Matryoshka** dimension shortening; and reading the **MTEB** leaderboard. All of that is in the [Embedding Models concept page](../03-Embedding-Models-for-Retrieval.md), with its [curated references](../03-Embedding-Models-for-Retrieval.references.md). Next: [vector databases & ANN indexes](../../04-Vector-Databases-and-ANN-Indexes/04-Vector-Databases-and-ANN-Indexes.md) make 'find the nearest vectors' fast at scale.